In [ ]:
# ============================================
# 1️⃣ Import Libraries
# ============================================
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing import image


In [ ]:

# ============================================
# 2️⃣ Define Paths
# ============================================
base_dir = "/kaggle/input/ai-vs-human-generated-dataset"
train_csv = os.path.join(base_dir, "train.csv")
test_csv = os.path.join(base_dir, "test.csv")

# Load CSVs
train_df = pd.read_csv(train_csv)
test_df = pd.read_csv(test_csv)

# Inspect CSV
print("Train CSV sample:")
print(train_df.head())

# Rename for clarity
train_df.rename(columns={'file_name': 'image_path'}, inplace=True)

# Create absolute paths for images
train_df['image_path'] = train_df['image_path'].apply(lambda x: os.path.join(base_dir, x))

# Verify sample path
print("\nSample image path:", train_df.iloc[0]['image_path'])
print("File exists:", os.path.exists(train_df.iloc[0]['image_path']))


In [ ]:

# ============================================
# 3️⃣ Split Train / Validation Sets
# ============================================
train_df, val_df = train_test_split(
    train_df,
    test_size=0.2,
    stratify=train_df['label'],
    random_state=42
)

print(f"\nTraining samples: {len(train_df)}, Validation samples: {len(val_df)}")

#  Convert labels to string (Fixes class_mode="binary" error)
train_df['label'] = train_df['label'].astype(str)
val_df['label'] = val_df['label'].astype(str)



In [ ]:

# ============================================
# 4️⃣ Image Data Generators
# ============================================
IMG_SIZE = (224, 224)
BATCH_SIZE = 32

train_datagen = ImageDataGenerator(
    rescale=1./255,
    horizontal_flip=True,
    rotation_range=15,
    zoom_range=0.1,
    brightness_range=[0.8,1.2]
)

val_datagen = ImageDataGenerator(rescale=1./255)

train_generator = train_datagen.flow_from_dataframe(
    dataframe=train_df,
    x_col='image_path',
    y_col='label',
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='binary',
    shuffle=True
)

val_generator = val_datagen.flow_from_dataframe(
    dataframe=val_df,
    x_col='image_path',
    y_col='label',
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='binary',
    shuffle=False
)


In [ ]:

# ============================================
# 5️⃣ Load Pretrained ResNet50 (ImageNet)
# ============================================
base_model = ResNet50(
    weights='imagenet',
    include_top=False,
    input_shape=(IMG_SIZE[0], IMG_SIZE[1], 3)
)

# Freeze base layers
for layer in base_model.layers:
    layer.trainable = False

# Add custom classification layers
x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dropout(0.5)(x)
predictions = Dense(1, activation='sigmoid')(x)

model = Model(inputs=base_model.input, outputs=predictions)

model.compile(
    optimizer=Adam(learning_rate=1e-4),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model.summary()


In [ ]:

# ============================================
# 6️⃣ Callbacks
# ============================================
callbacks = [
    EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True),
    ModelCheckpoint('resnet50_ai_vs_real_best.h5', monitor='val_loss', save_best_only=True)
]


In [ ]:

# ============================================
# 7️⃣ Train Model
# ============================================
EPOCHS = 10

history = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=EPOCHS,
    callbacks=callbacks
)


In [ ]:

# ============================================
# 8️⃣ Plot Training Curves
# ============================================
plt.figure(figsize=(12,5))
plt.subplot(1,2,1)
plt.plot(history.history['accuracy'], label='Train Accuracy')
plt.plot(history.history['val_accuracy'], label='Val Accuracy')
plt.title('Accuracy')
plt.legend()

plt.subplot(1,2,2)
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Val Loss')
plt.title('Loss')
plt.legend()
plt.show()


In [ ]:

# ============================================
# 9️⃣ Save Final Model
# ============================================
model.save("resnet50_ai_vs_real_final.h5")
print("✅ Model saved as resnet50_ai_vs_real_final.h5")


In [ ]:

# ============================================
# 🔟 Predict on a New Image
# ============================================
def predict_image(img_path, model, target_size=(224,224)):
    img = image.load_img(img_path, target_size=target_size)
    img_array = image.img_to_array(img) / 255.0
    img_array = np.expand_dims(img_array, axis=0)
    prob = model.predict(img_array)[0][0]
    label = 'AI-generated' if prob > 0.5 else 'Real'
    return label, prob

# Example usage:
example_image = train_df.iloc[0]['image_path']
label, prob = predict_image(example_image, model)
print(f"\nPrediction: {label}, Probability: {prob:.4f}")
